# 00 · Catálogo y casos para las pruebas RAG

Se leen directamente los archivos:

- `04_limpieza/06_Finales/Canonico_FINAL.csv`
- `06_grafos/catalogo_articulos.csv`

El código genera **solo dos archivos**:

- `auditoria_catalogo.json`: comprobaciones, incidencias, coincidencias textuales, huellas y registro de ejecuciones.
- `consultas_para_revision.csv`: las 24 consultas propuestas.


## 1. Localizar el repositorio y leer los originales

La búsqueda comprueba el directorio de trabajo y sus carpetas superiores, sin recorrer otras ramas.

In [1]:
import csv
import io
import json
import hashlib
import re
import sys
import statistics
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

VERSION = "AUDITORIA_CATALOGO_RAG_V2_07RAG"

# Rutas relativas a la raíz del repositorio, según la organización actual.
FUENTES = {
    "canonico": {
        "ruta": "04_limpieza/06_Finales/Canonico_FINAL.csv",
        "sha256_bytes": "83547be2ffabaf81877332a3963ebb28b7b25dd7d5596296f289180e60090bc5",
        "sha256_contenido_csv": "c43adadcdf92e098dab2882566b8e95cd5e92d068ddd053b6b83cd92bb1aa112",
        "columnas": ["indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
                     "ISBN", "ISSN", "Doi", "URL", "Area", "SubArea", "Keywords", "Abstract"]
    },
    "catalogo": {
        "ruta": "06_grafos/catalogo_articulos.csv",
        "sha256_bytes": "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b",
        "sha256_contenido_csv": "343bb0e20338b7660a0225af43d4481cc2e67b9cb7351d8105b2945670e479aa",
        "columnas": ["article_id", "titulo", "autores", "anio", "area", "keywords",
                     "abstract", "doi", "url"]
    }
}


def sha256(datos):
    return hashlib.sha256(datos).hexdigest()


def leer_csv_bytes(datos, nombre):
    # Mantener los campos como texto; no convertir años, identificadores ni DOI.
    with io.StringIO(datos.decode("utf-8-sig"), newline="") as f:
        lector = csv.DictReader(f)
        columnas = lector.fieldnames or []
        filas = list(lector)
    if not columnas or len(columnas) != len(set(columnas)):
        raise ValueError(f"Encabezados ausentes o repetidos: {nombre}")
    if any(None in r or any(v is None for v in r.values()) for r in filas):
        raise ValueError(f"Número irregular de campos CSV: {nombre}")
    if not filas:
        raise ValueError(f"El archivo no contiene registros: {nombre}")
    return columnas, filas


def huella_contenido(columnas, filas):
    contenido = json.dumps({"columnas": columnas, "filas": filas},
                           ensure_ascii=False, separators=(",", ":"))
    return sha256(contenido.encode("utf-8"))


actual = Path.cwd().resolve()
RAIZ = next((p for p in (actual, *actual.parents)
             if (p / "04_limpieza").is_dir() and (p / "06_grafos").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError(
        "No se encontró la raíz de Tesis_Multimodelo. Abre esta libreta dentro de 07_RAG "
        "y ejecuta el kernel desde esa carpeta o desde la raíz del repositorio."
    )

SALIDA = RAIZ / "07_RAG"
entradas, identidad = {}, {}
for rol, especificacion in FUENTES.items():
    ruta = RAIZ / especificacion["ruta"]
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta el archivo: {ruta}. Revisa su nombre; no copies los datos a 07_RAG.")
    datos = ruta.read_bytes()
    columnas, filas = leer_csv_bytes(datos, ruta.name)
    h_bytes = sha256(datos)
    h_campos = huella_contenido(columnas, filas)
    if columnas != especificacion["columnas"] or h_campos != especificacion["sha256_contenido_csv"]:
        raise ValueError(
            f"El contenido de {ruta.name} difiere del archivo auditado. "
            "No se modificó ningún archivo. Revisa la versión antes de continuar; "
            "no reemplaces la huella esperada para ocultar la diferencia."
        )
    if h_bytes != especificacion["sha256_bytes"]:
        print("AVISO: bytes distintos, pero campos CSV idénticos:", ruta.name)
    entradas[rol] = filas
    identidad[rol] = {
        "ruta_relativa": especificacion["ruta"], "bytes": len(datos), "filas": len(filas),
        "columnas": columnas, "sha256_bytes": h_bytes, "sha256_contenido_csv": h_campos,
        "bytes_iguales_al_adjunto": h_bytes == especificacion["sha256_bytes"]
    }
    print(rol, "| filas:", len(filas), "| columnas:", len(columnas), "| SHA256:", h_bytes)

canonico = entradas["canonico"]
catalogo = entradas["catalogo"]
print("Raíz:", RAIZ)
print("Resultados:", SALIDA)
print("Python:", sys.version.split()[0])
print("Solo se han leído los archivos; los originales permanecen intactos.")


canonico | filas: 1151 | columnas: 14 | SHA256: 83547be2ffabaf81877332a3963ebb28b7b25dd7d5596296f289180e60090bc5
catalogo | filas: 519 | columnas: 9 | SHA256: 74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b
Raíz: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo
Resultados: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG
Python: 3.11.16
Solo se han leído los archivos; los originales permanecen intactos.


## 2. Identificadores, campos y autoría registrada

Se contrasta `article_id` con `indice`, sin reasignar identificadores. Se comparan título, año, área, palabras clave, resumen, DOI y URL **exactamente**, sin limpiar ni completar campos.

La lista `autores` se separa por punto y coma para compararla con los nombres de `Autor_norm`. Eso interpreta el delimitador del catálogo; no cambia la escritura de los nombres.

In [2]:
por_indice = defaultdict(list)
for numero, fila in enumerate(canonico, start=1):
    por_indice[fila["indice"]].append((numero, fila))

mapa_campos = {"Titulo": "titulo", "Año": "anio", "Area": "area", "Keywords": "keywords",
               "Abstract": "abstract", "Doi": "doi", "URL": "url"}
incidencias = []

def anotar(nivel, tipo, article_id, campo, detalle):
    incidencias.append({"nivel": nivel, "tipo": tipo, "article_id": article_id,
                       "campo": campo, "detalle": detalle})

ids_catalogo = [r["article_id"] for r in catalogo]
conteos_id = Counter(ids_catalogo)
for identificador, n in conteos_id.items():
    if not identificador.strip() or n != 1:
        anotar("bloqueante", "identificador_no_unico", identificador, "article_id", str(n))
    if not re.fullmatch(r"[1-9][0-9]*", identificador):
        anotar("bloqueante", "identificador_no_entero_positivo", identificador, "article_id", identificador)
for identificador in sorted(set(por_indice) ^ set(ids_catalogo)):
    anotar("bloqueante", "identificador_sin_correspondencia", identificador, "indice/article_id",
           "El identificador aparece solo en uno de los archivos.")

comparaciones = 0
for r in catalogo:
    aid = r["article_id"]
    fuentes = por_indice.get(aid, [])
    if not fuentes:
        continue
    for campo_canonico, campo_catalogo in mapa_campos.items():
        valores = {f[campo_canonico] for _, f in fuentes}
        comparaciones += 1
        if len(valores) != 1:
            anotar("bloqueante", "conflicto_interno_canonico", aid, campo_canonico,
                   json.dumps(sorted(valores), ensure_ascii=False))
        if valores != {r[campo_catalogo]}:
            anotar("bloqueante", "diferencia_canonico_catalogo", aid, campo_catalogo,
                   "Los campos no coinciden literalmente.")
    autores_fuente = {f["Autor_norm"] for _, f in fuentes}
    lista_autores = [a.strip() for a in r["autores"].split(";") if a.strip()]
    if autores_fuente != set(lista_autores) or len(lista_autores) != len(set(lista_autores)):
        anotar("bloqueante", "diferencia_autores", aid, "autores",
               json.dumps({"canonico": sorted(autores_fuente), "catalogo": lista_autores}, ensure_ascii=False))
    if r["anio"] not in {"2024", "2025"}:
        anotar("bloqueante", "anio_fuera_periodo", aid, "anio", r["anio"])
    for campo in ("titulo", "autores", "abstract"):
        if not r[campo].strip():
            anotar("advertencia", "campo_vacio", aid, campo, "No se completa automáticamente.")
    for campo in ("doi", "keywords"):
        if not r[campo].strip():
            anotar("informativa", "campo_vacio", aid, campo, "No impide representar el resumen disponible.")

pares = Counter((r["indice"], r["Autor_norm"]) for r in canonico)
for (aid, autor), cantidad in pares.items():
    if cantidad > 1:
        anotar("advertencia", "par_articulo_autor_repetido", aid, "Autor_norm",
               f"{autor}: {cantidad} filas; revisar afiliaciones antes de cualquier decisión.")

print("Comparaciones bibliográficas:", comparaciones)
print("Coincidencia de conjuntos de identificadores:", set(por_indice) == set(ids_catalogo))
print("Incidencias bloqueantes:", sum(i["nivel"] == "bloqueante" for i in incidencias))

Comparaciones bibliográficas: 3633
Coincidencia de conjuntos de identificadores: True
Incidencias bloqueantes: 0


## 3. Textos repetidos y perfil descriptivo

Dos identificadores con el mismo resumen no se fusionan: pueden corresponder a versiones u otros casos bibliográficos que requieren revisión. Se asigna un grupo auxiliar de coincidencia **textual exacta**; no es un identificador de obra ni una decisión de deduplicación.

Las longitudes se cuentan por palabras separadas por espacios, **no por tokens del modelo**. No se estima el idioma, la calidad del resumen ni su coincidencia con el texto publicado.

In [3]:
grupos_abstract = defaultdict(list)
grupos_titulo = defaultdict(list)
grupos_doi = defaultdict(list)
for r in catalogo:
    if r["abstract"].strip():
        grupos_abstract[r["abstract"]].append(r["article_id"])
    if r["titulo"].strip():
        grupos_titulo[r["titulo"]].append(r["article_id"])
    if r["doi"].strip():
        grupos_doi[r["doi"]].append(r["article_id"])

por_id = {r["article_id"]: r for r in catalogo}
coincidencias = []
repetidos_abstract = [ids for ids in grupos_abstract.values() if len(ids) > 1]
for numero, ids in enumerate(repetidos_abstract, start=1):
    grupo = f"ABS-{numero:03d}"
    for aid in ids:
        r = por_id[aid]
        coincidencias.append({"grupo_texto_identico": grupo, "article_id": aid,
                              "titulo": r["titulo"], "anio": r["anio"], "doi": r["doi"],
                              "estado_revision": "pendiente", "decision": "", "observacion": ""})
        anotar("advertencia", "abstract_identico_otro_id", aid, "abstract",
               "Identificadores con texto idéntico: " + "; ".join(ids) + ". No se fusionan.")
for ids in grupos_doi.values():
    if len(ids) > 1:
        for aid in ids:
            anotar("advertencia", "doi_identico_otro_id", aid, "doi", "; ".join(ids))

longitudes = [len(r["abstract"].split()) for r in catalogo if r["abstract"].strip()]
resumen = {
    "version_auditoria": VERSION,
    "filas_canonico": len(canonico),
    "filas_catalogo": len(catalogo),
    "identificadores_articulo": len(set(ids_catalogo)),
    "nombres_autor_distintos_exactos": len({r["Autor_norm"] for r in canonico}),
    "pares_articulo_autor_distintos": len(pares),
    "comparaciones_bibliograficas": comparaciones,
    "incidencias_bloqueantes": sum(i["nivel"] == "bloqueante" for i in incidencias),
    "articulos_con_resumen": len(longitudes),
    "textos_abstract_distintos_exactos": len(grupos_abstract),
    "grupos_abstract_identico": len(repetidos_abstract),
    "grupos_titulo_identico": sum(len(ids) > 1 for ids in grupos_titulo.values()),
    "grupos_doi_identico_no_vacio": sum(len(ids) > 1 for ids in grupos_doi.values()),
    "vacios_catalogo": {c: sum(not r[c].strip() for r in catalogo) for c in catalogo[0]},
    "articulos_por_anio": dict(sorted(Counter(r["anio"] for r in catalogo).items())),
    "articulos_por_area": dict(sorted(Counter(r["area"] for r in catalogo).items())),
    "resumen_palabras": {"minimo": min(longitudes) if longitudes else None,
                         "mediana": statistics.median(longitudes) if longitudes else None,
                         "maximo": max(longitudes) if longitudes else None},
    "alcance": "Integridad y correspondencia internas. No verifica autenticidad editorial, autoría completa, idioma ni pertinencia científica."
}
print(json.dumps(resumen, ensure_ascii=False, indent=2))
print("\nPares/grupos de resúmenes idénticos:", repetidos_abstract)

{
  "version_auditoria": "AUDITORIA_CATALOGO_RAG_V2_07RAG",
  "filas_canonico": 1151,
  "filas_catalogo": 519,
  "identificadores_articulo": 519,
  "nombres_autor_distintos_exactos": 628,
  "pares_articulo_autor_distintos": 1151,
  "comparaciones_bibliograficas": 3633,
  "incidencias_bloqueantes": 0,
  "articulos_con_resumen": 519,
  "textos_abstract_distintos_exactos": 514,
  "grupos_abstract_identico": 5,
  "grupos_titulo_identico": 4,
  "grupos_doi_identico_no_vacio": 0,
  "vacios_catalogo": {
    "article_id": 0,
    "titulo": 0,
    "autores": 0,
    "anio": 0,
    "area": 0,
    "keywords": 5,
    "abstract": 0,
    "doi": 25,
    "url": 0
  },
  "articulos_por_anio": {
    "2024": 236,
    "2025": 283
  },
  "articulos_por_area": {
    "CC": 100,
    "IA": 182,
    "ISBD": 54,
    "RS": 27,
    "SIAV": 100,
    "TC": 56
  },
  "resumen_palabras": {
    "minimo": 23,
    "mediana": 150,
    "maximo": 443
  },
  "alcance": "Integridad y correspondencia internas. No verifica autent

## 4. Consultas propuestas incluidas en la libreta

**12 necesidades de información, con dos formulaciones (español e inglés), y una división propuesta de 4 necesidades de desarrollo y 8 de comparación**.


In [4]:
CONSULTAS_JSON_ORIGINAL = r'''{
  "version": "CONSULTAS_RAG_BORRADOR_V1",
  "estado": "NO_CONGELADO_NO_ES_GOLD_STANDARD",
  "catalogo_sha256": "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b",
  "origen": "Propuesta del asistente basada en los resúmenes del catálogo proporcionado; no generada por el recuperador ni por una inferencia local.",
  "advertencias": [
    "Las referencias orientadoras no son una lista exhaustiva de artículos relevantes ni qrels aprobados.",
    "No enviar títulos, identificadores orientadores ni fragmentos de esta guía como parte de la consulta de búsqueda.",
    "Las dos formulaciones lingüísticas de una necesidad pertenecen al mismo grupo; no son observaciones independientes.",
    "La división 4 desarrollo / 8 comparación es propuesta; revisar y fijar antes de evaluar y no ajustar sobre resultados de comparación.",
    "No se declara independencia respecto del conocimiento general del corpus; las consultas fueron redactadas tras revisar textos del catálogo."
  ],
  "necesidades": [
    {
      "necesidad_id": "N01",
      "division_propuesta": "desarrollo",
      "tema": "RAG para documentación técnica",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N01-ES",
          "idioma": "es",
          "texto": "¿Qué trabajos permiten consultar documentación técnica de un sistema mediante preguntas en lenguaje natural y fundamentan la respuesta en material recuperado?"
        },
        {
          "query_id": "N01-EN",
          "idioma": "en",
          "texto": "Which studies support natural-language questions about system documentation and ground the answers in retrieved reference material?"
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "117",
          "titulo": "A retrieval-augmented-generation pipeline to help users query system-provided documentation",
          "fragmento_literal_abstract": "A retrieval-augmented-generation pipeline is proposed to let users ask natural-language questions over system-provided Debian documentation.",
          "sha256_abstract_utf8": "837d44ab7269a1bfc5f5a2237f8763d1f15ac6623f03073e8ea5643679423e86"
        }
      ]
    },
    {
      "necesidad_id": "N02",
      "division_propuesta": "desarrollo",
      "tema": "Informes mamográficos en español",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N02-ES",
          "idioma": "es",
          "texto": "Busco recursos y métodos de procesamiento del lenguaje para organizar o clasificar informes de mamografía en español según BI-RADS."
        },
        {
          "query_id": "N02-EN",
          "idioma": "en",
          "texto": "I am looking for language-processing resources and methods to organize or classify Spanish mammography reports according to BI-RADS."
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "129",
          "titulo": "Automatic Classification of BI-RADS in Spanish Radiology Reports Using Transformers and Traditional Machine Learning Approaches",
          "fragmento_literal_abstract": "Mammography reports are crucial for the early detection of breast cancer; however, their unstructured format and high volume pose challenges for automated evaluation.",
          "sha256_abstract_utf8": "7b225b227d012cb1aa420b84c83a90a89132c873e42c3663271a7ee0379b2f7d"
        },
        {
          "article_id": "314",
          "titulo": "Mammography reporting dataset with BI-RADS system for natural language processing applications: Addressing public data gaps in Spanish",
          "fragmento_literal_abstract": "The article presents a Spanish mammography-report dataset with 4,357 records and 15 variables, including complete and sectioned report text, BI-RADS labels and selected metadata.",
          "sha256_abstract_utf8": "3fa6ae41d2fda0b5d4d1778d827c4fb62a9eb6e7904e7fe5d51a5f1154e5df58"
        }
      ]
    },
    {
      "necesidad_id": "N03",
      "division_propuesta": "desarrollo",
      "tema": "Garantías de privacidad en entrenamiento",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N03-ES",
          "idioma": "es",
          "texto": "¿Cómo se estudian las garantías de privacidad del entrenamiento por descenso de gradiente cuando se recortan los gradientes?"
        },
        {
          "query_id": "N03-EN",
          "idioma": "en",
          "texto": "How are privacy guarantees studied for gradient-descent training when gradients are clipped?"
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "355",
          "titulo": "On the Privacy Guarantees of Differentially Private Stochastic Gradient Descent",
          "fragmento_literal_abstract": "Differentially Private Stochastic Gradient Descent (DP-SGD) is a widely adopted algorithm for privately training machine learning models.",
          "sha256_abstract_utf8": "864fd851969eb0306d7e7c70d131137b03744ebecb00c9d5b19ba6548d9cb4d8"
        }
      ]
    },
    {
      "necesidad_id": "N04",
      "division_propuesta": "desarrollo",
      "tema": "Representación de textos mediante grafos",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N04-ES",
          "idioma": "es",
          "texto": "Busco herramientas que conviertan documentos textuales en grafos para apoyar la recuperación de información o la clasificación de temas."
        },
        {
          "query_id": "N04-EN",
          "idioma": "en",
          "texto": "I am looking for tools that turn text documents into graphs to support information retrieval or topic classification."
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "324",
          "titulo": "text2graphAPI: A library to transform text documents into different graph representations",
          "fragmento_literal_abstract": "This paper introduces a new Python API called text2graphAPI.",
          "sha256_abstract_utf8": "1cb96d289cb73a0c5948ffaf512b1ec47c92ae11223058c6c0e5601296a081a9"
        }
      ]
    },
    {
      "necesidad_id": "N05",
      "division_propuesta": "comparacion",
      "tema": "Síntesis de programas",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N05-ES",
          "idioma": "es",
          "texto": "¿Qué investigaciones estudian la generación automática de programas a partir de requisitos expresados en lenguaje natural y sus dificultades?"
        },
        {
          "query_id": "N05-EN",
          "idioma": "en",
          "texto": "Which studies examine automatic program generation from natural-language requirements and its challenges?"
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "285",
          "titulo": "A Systematic Literature Review of 10 years of Research on Program Synthesis and Natural Language Processing",
          "fragmento_literal_abstract": "Program synthesis is the process of automatically generating software from a requirement specification.",
          "sha256_abstract_utf8": "952dcfbab7430f79f7193fc51bcd4a84d2ca337de965e4726afcef254a322c75"
        }
      ]
    },
    {
      "necesidad_id": "N06",
      "division_propuesta": "comparacion",
      "tema": "Integración de datos para estimación de software",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N06-ES",
          "idioma": "es",
          "texto": "¿Cómo se puede comprobar si es adecuado combinar datos de distintas fuentes para mejorar la estimación del esfuerzo de desarrollo de software?"
        },
        {
          "query_id": "N06-EN",
          "idioma": "en",
          "texto": "How can the suitability of combining data from different sources be checked to improve software development effort estimation?"
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "316",
          "titulo": "Merging Distinct Sources Databases to Improve Software Estimation Models",
          "fragmento_literal_abstract": "Context.",
          "sha256_abstract_utf8": "0fbbf3817d678703e45095452aaf683ac8a48426b1231b66aed5a7172b02b65f"
        }
      ]
    },
    {
      "necesidad_id": "N07",
      "division_propuesta": "comparacion",
      "tema": "Monitoreo de sistemas eléctricos",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N07-ES",
          "idioma": "es",
          "texto": "Busco plataformas que procesen mediciones de un sistema eléctrico y permitan analizar oscilaciones y grupos de nodos con comportamiento coherente."
        },
        {
          "query_id": "N07-EN",
          "idioma": "en",
          "texto": "I am looking for platforms that process power-system measurements and support the analysis of oscillations and groups of nodes with coherent behavior."
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "336",
          "titulo": "Web-based wide-area monitoring platform for ringdown and clustering analytics in power systems",
          "fragmento_literal_abstract": "This paper introduces an open-source research platform for monitoring the Mexican interconnected power grid, allowing real-time processing and information extraction of the grid's dynamic condition.",
          "sha256_abstract_utf8": "745bd77490eb0ce52f8ed8f67d75627c9e082454d15be9eb9b3316c5f3bb2346"
        }
      ]
    },
    {
      "necesidad_id": "N08",
      "division_propuesta": "comparacion",
      "tema": "Aprendizaje científico y ecuaciones astronómicas",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N08-ES",
          "idioma": "es",
          "texto": "¿Qué estudios usan modelos neuronales de ecuaciones diferenciales para predecir la estructura de enanas blancas y analizar cuándo falla el pronóstico?"
        },
        {
          "query_id": "N08-EN",
          "idioma": "en",
          "texto": "Which studies use neural differential-equation models to predict white-dwarf structure and analyze when forecasting fails?"
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "1",
          "titulo": "A comparative study of NeuralODE and Universal ODE approaches to solving Chandrasekhar White Dwarf equation",
          "fragmento_literal_abstract": "In this study, we apply two pillars of Scientific Machine Learning: Neural Ordinary Differential Equations (Neural ODEs) and Universal Differential Equations (UDEs) to the Chandrasekhar White Dwarf Equation (CWDE).",
          "sha256_abstract_utf8": "2af5391a04dfe47b01cd7eaaeba1fae7064cb875d7c1eba9dbe12357ba1ebb04"
        }
      ]
    },
    {
      "necesidad_id": "N09",
      "division_propuesta": "comparacion",
      "tema": "Optimización de portafolios con ESG",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N09-ES",
          "idioma": "es",
          "texto": "¿Cómo se incorpora un criterio ambiental, social y de gobernanza al equilibrio entre riesgo y rendimiento de un portafolio?"
        },
        {
          "query_id": "N09-EN",
          "idioma": "en",
          "texto": "How is an environmental, social, and governance criterion incorporated into the risk-return trade-off of a portfolio?"
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "4",
          "titulo": "A Hierarchical Approach to a Tri-Objective Portfolio Optimization Problem Considering an ESG Index",
          "fragmento_literal_abstract": "Traditional portfolio construction primarily revolves around a bi-objective approach, focusing on minimizing portfolio variance while maximizing expected returns.",
          "sha256_abstract_utf8": "af57dd0f1a37605d43afec5a414998a8fad13d76011ba96482bd8bc922a772ef"
        }
      ]
    },
    {
      "necesidad_id": "N10",
      "division_propuesta": "comparacion",
      "tema": "Aprendizaje contrastivo de grafos",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N10-ES",
          "idioma": "es",
          "texto": "¿Qué métodos incorporan información topológica de estructuras de orden superior para mejorar el aprendizaje contrastivo de grafos?"
        },
        {
          "query_id": "N10-EN",
          "idioma": "en",
          "texto": "Which methods incorporate topological information from higher-order structures to improve graph contrastive learning?"
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "271",
          "titulo": "TopoGCL: Topological Graph Contrastive Learning",
          "fragmento_literal_abstract": "Graph contrastive learning (GCL) has recently emerged as a new concept which allows for capitalizing on the strengths of graph neural networks (GNNs) to learn rich representations in a wide variety of applications which involve abundant unlabeled information.",
          "sha256_abstract_utf8": "fae655dbf12bb32bbe7b500e88cf87005816bd633c788178b8fbca2e0876982e"
        }
      ]
    },
    {
      "necesidad_id": "N11",
      "division_propuesta": "comparacion",
      "tema": "Esteganografía paralela",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N11-ES",
          "idioma": "es",
          "texto": "Busco técnicas que oculten información en imágenes mediante una clave secreta y aprovechen procesamiento paralelo para acelerar la codificación."
        },
        {
          "query_id": "N11-EN",
          "idioma": "en",
          "texto": "I am looking for techniques that hide information in images using a secret key and exploit parallel processing to speed up encoding."
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "337",
          "titulo": "A Parallel Steganography Algorithm Using a New Secret Key Generation",
          "fragmento_literal_abstract": "Computation has mostly used parallelism for high-performance computation, and multicore processors have gained popularity because they offer better processing time.",
          "sha256_abstract_utf8": "54cbff9db8f8bd697336cc4d74d8095a4103d3311ab779adf71d51eeb89b5d18"
        }
      ]
    },
    {
      "necesidad_id": "N12",
      "division_propuesta": "comparacion",
      "tema": "Juegos de persecución en grafos",
      "estado": "propuesta_pendiente_revision_humana",
      "consultas": [
        {
          "query_id": "N12-ES",
          "idioma": "es",
          "texto": "¿Qué trabajos analizan el número de policías necesario para capturar ladrones en grafos de fichas de caminos o estrellas?"
        },
        {
          "query_id": "N12-EN",
          "idioma": "en",
          "texto": "Which studies analyze the number of cops needed to capture robbers on token graphs of paths or stars?"
        }
      ],
      "orientacion_no_enviar_al_recuperador": [
        {
          "article_id": "482",
          "titulo": "Cops and Robbers on Token Graphs",
          "fragmento_literal_abstract": "Let G = (V, E) be a graph and k a positive integer such that k ≤ |V|.",
          "sha256_abstract_utf8": "9da1d65e19285d427caa198e77a08106464d44d6449b62993d21e9c2fc5a7f00"
        }
      ]
    }
  ]
}
'''

bytes_consultas = CONSULTAS_JSON_ORIGINAL.encode("utf-8")
CONSULTAS_SHA_ESPERADO = "728c94409766cf93289de39b156cf03b93a3d1ab6542be82f3c656880bf795b7"
if sha256(bytes_consultas) != CONSULTAS_SHA_ESPERADO:
    raise ValueError("Cambió el borrador incluido en la libreta; no se ha guardado ningún resultado.")
consultas = json.loads(CONSULTAS_JSON_ORIGINAL)
filas_consultas = []
for necesidad in consultas["necesidades"]:
    referencias = necesidad["orientacion_no_enviar_al_recuperador"]
    for referencia in referencias:
        aid = referencia["article_id"]
        if aid not in por_id:
            raise ValueError(f"Artículo orientador ausente: {aid}")
        texto = por_id[aid]["abstract"]
        if referencia["fragmento_literal_abstract"] not in texto:
            raise ValueError(f"Fragmento orientador distinto del original: {aid}")
        if sha256(texto.encode("utf-8")) != referencia["sha256_abstract_utf8"]:
            raise ValueError(f"Cambió el resumen orientador: {aid}")
    for q in necesidad["consultas"]:
        filas_consultas.append({
            "query_id": q["query_id"], "necesidad_id": necesidad["necesidad_id"],
            "division_propuesta": necesidad["division_propuesta"], "idioma": q["idioma"],
            "consulta_propuesta": q["texto"],
            "articulos_orientadores_no_exhaustivos": "; ".join(r["article_id"] for r in referencias),
            "revision_humana": "pendiente", "comentario": ""
        })

# Los documentos se conservan solo en memoria. No se genera otra copia del catálogo.
documentos = [{"article_id": r["article_id"], "texto": r["abstract"],
               "campo_origen": "abstract", "registro_csv_1_based": n,
               "sha256_texto_utf8": sha256(r["abstract"].encode("utf-8"))}
              for n, r in enumerate(catalogo, start=1) if r["abstract"].strip()]

print("Necesidades:", len(consultas["necesidades"]), "| Consultas propuestas:", len(filas_consultas))
print("División propuesta:", dict(Counter(n["division_propuesta"] for n in consultas["necesidades"])))
print("Resúmenes disponibles en memoria:", len(documentos))
print("No se han generado embeddings ni aprobado consultas automáticamente.")
for r in filas_consultas[:4]:
    print(r["query_id"], "|", r["consulta_propuesta"])


Necesidades: 12 | Consultas propuestas: 24
División propuesta: {'desarrollo': 4, 'comparacion': 8}
Resúmenes disponibles en memoria: 519
No se han generado embeddings ni aprobado consultas automáticamente.
N01-ES | ¿Qué trabajos permiten consultar documentación técnica de un sistema mediante preguntas en lenguaje natural y fundamentan la respuesta en material recuperado?
N01-EN | Which studies support natural-language questions about system documentation and ground the answers in retrieved reference material?
N02-ES | Busco recursos y métodos de procesamiento del lenguaje para organizar o clasificar informes de mamografía en español según BI-RADS.
N02-EN | I am looking for language-processing resources and methods to organize or classify Spanish mammography reports according to BI-RADS.


## 5. Guardar archivos

`auditoria_catalogo.json` reúne el resumen, incidencias, grupos de texto idéntico, huellas y el historial de ejecuciones. Al repetir la celda con las mismas entradas se actualiza ese mismo informe, sin crear copias numeradas ni carpetas adicionales.

`consultas_para_revision.csv` se crea solamente si no existe. **Si ya existe, se lee y se conserva tal como está**, incluidos los comentarios y revisiones añadidos manualmente. El informe registra su huella y sus filas actuales; la libreta no aprueba ni restaura automáticamente las consultas.


In [5]:
def como_json(objeto):
    return (json.dumps(objeto, ensure_ascii=False, indent=2) + "\n").encode("utf-8")


def como_csv(filas, columnas):
    memoria = io.StringIO(newline="")
    escritor = csv.DictWriter(memoria, fieldnames=columnas, lineterminator="\n")
    escritor.writeheader()
    escritor.writerows(filas)
    return memoria.getvalue().encode("utf-8-sig")


# Comprobar de nuevo las entradas antes de escribir cualquier resultado.
for rol, especificacion in FUENTES.items():
    if sha256((RAIZ / especificacion["ruta"]).read_bytes()) != identidad[rol]["sha256_bytes"]:
        raise RuntimeError("Una entrada cambió durante la auditoría: " + especificacion["ruta"])

SALIDA.mkdir(exist_ok=True)
ruta_auditoria = SALIDA / "auditoria_catalogo.json"
ruta_consultas = SALIDA / "consultas_para_revision.csv"
columnas_consultas = ["query_id", "necesidad_id", "division_propuesta", "idioma",
                     "consulta_propuesta", "articulos_orientadores_no_exhaustivos",
                     "revision_humana", "comentario"]

# Conservar el historial del informe, sin sobrescribir resultados de otro catálogo.
historial = []
if ruta_auditoria.exists():
    anterior = json.loads(ruta_auditoria.read_text(encoding="utf-8-sig"))
    if anterior.get("version") != VERSION:
        raise ValueError("auditoria_catalogo.json pertenece a otra versión; consérvala antes de reemplazarla.")
    for rol in FUENTES:
        previo = anterior.get("fuentes", {}).get(rol, {}).get("sha256_contenido_csv")
        if previo != identidad[rol]["sha256_contenido_csv"]:
            raise ValueError("El informe existente corresponde a otras entradas; no se sobrescribe.")
    historial = anterior.get("ejecuciones", [])
    if not isinstance(historial, list):
        raise ValueError("El historial del informe no tiene el formato esperado.")

# Nunca sobrescribir el CSV editable: conservar su contenido y registrar la versión leída.
consultas_ya_existian = ruta_consultas.exists()
if consultas_ya_existian:
    contenido_csv = ruta_consultas.read_bytes()
    columnas_actuales, consultas_actuales = leer_csv_bytes(contenido_csv, ruta_consultas.name)
    if columnas_actuales != columnas_consultas:
        raise ValueError("Cambió el encabezado de consultas_para_revision.csv. No se sobrescribió el archivo.")
    ids_actuales = [r["query_id"] for r in consultas_actuales]
    ids_esperados = {r["query_id"] for r in filas_consultas}
    if len(ids_actuales) != len(set(ids_actuales)) or set(ids_actuales) != ids_esperados:
        raise ValueError("Revisar los identificadores de consulta; se esperaban los mismos 24 identificadores únicos.")
else:
    contenido_csv = como_csv(filas_consultas, columnas_consultas)
    consultas_actuales = filas_consultas

historial.append({
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "sha256_entradas": {k: v["sha256_bytes"] for k, v in identidad.items()},
    "sha256_consultas_csv": sha256(contenido_csv),
    "consultas_csv_conservado_sin_sobrescribir": consultas_ya_existian
})

informe = {
    "version": VERSION,
    "fuentes": identidad,
    "resumen": resumen,
    "incidencias": incidencias,
    "coincidencias_abstract_para_revision": coincidencias,
    "borrador_consultas_incluido": consultas,
    "sha256_borrador_consultas_original": sha256(bytes_consultas),
    "consultas_csv": {
        "archivo": ruta_consultas.name, "sha256_bytes": sha256(contenido_csv),
        "filas": consultas_actuales,
        "nota": "Estado de revisión según el CSV; esta libreta no valida ni aprueba pertinencia."
    },
    "ejecuciones": historial,
    "limites": [
        "Solo consistencia interna entre los CSV; no se verifican editoriales ni autoría completa.",
        "Se conserva cada identificador; resúmenes iguales no se fusionan automáticamente.",
        "No se midieron idioma ni longitud en tokens; el perfil usa palabras separadas por espacios.",
        "Los artículos orientadores no son una lista exhaustiva de relevantes ni qrels aprobados."
    ],
    "modelos_ejecutados": [], "embeddings_generados": False, "grafo_modificado": False
}

# Dos archivos persistentes; ninguna carpeta de datos o resultados adicional.
if not consultas_ya_existian:
    with ruta_consultas.open("xb") as f:
        f.write(contenido_csv)
ruta_auditoria.write_bytes(como_json(informe))

print("AUDITORÍA FINALIZADA")
print("Filas canónico:", resumen["filas_canonico"])
print("Artículos:", resumen["identificadores_articulo"])
print("Resúmenes no vacíos:", resumen["articulos_con_resumen"])
print("Resúmenes distintos exactos:", resumen["textos_abstract_distintos_exactos"])
print("Comparaciones bibliográficas:", resumen["comparaciones_bibliograficas"])
print("Conflictos bloqueantes:", resumen["incidencias_bloqueantes"])
print("Grupos de resumen idéntico:", resumen["grupos_abstract_identico"])
print("Consultas:", len(consultas_actuales), "| estados registrados:",
      dict(Counter(r["revision_humana"] for r in consultas_actuales)))
print("Informe:", ruta_auditoria)
print("Consultas:", ruta_consultas,
      "(existente, conservado)" if consultas_ya_existian else "(creado)")
print("Dos archivos de salida en 07_RAG. CSV originales y grafos sin modificaciones.")
print("No se ejecutaron modelos ni búsquedas de recuperación.")
if resumen["incidencias_bloqueantes"]:
    raise RuntimeError("Revisar las incidencias del JSON antes de preparar vectores.")


AUDITORÍA FINALIZADA
Filas canónico: 1151
Artículos: 519
Resúmenes no vacíos: 519
Resúmenes distintos exactos: 514
Comparaciones bibliográficas: 3633
Conflictos bloqueantes: 0
Grupos de resumen idéntico: 5
Consultas: 24 | estados registrados: {'pendiente': 24}
Informe: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\auditoria_catalogo.json
Consultas: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\consultas_para_revision.csv (creado)
Dos archivos de salida en 07_RAG. CSV originales y grafos sin modificaciones.
No se ejecutaron modelos ni búsquedas de recuperación.
